In [ ]:
from pandas import Series, DataFrame
import pandas as pd
import numpy as np
import sys

np.set_printoptions(threshold=sys.maxsize)
from IPython.display import display

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
import platform

path = "c:/Windows/Fonts/malgun.ttf"
from matplotlib import font_manager, rc

if platform.system() == "Darwin":
    rc("font", family="AppleGothic")
elif platform.system() == "Windows":
    font_name = font_manager.FontProperties(fname=path).get_name()
    rc("font", family=font_name)
else:
    print("Unknown system... sorry~~~~")

In [3]:
import tensorflow as tf
import keras

Using TensorFlow backend.


In [4]:
from keras import models
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding
from keras.preprocessing import sequence
from keras.utils import np_utils
from keras.callbacks import ModelCheckpoint, EarlyStopping

In [5]:
title_basic = pd.read_csv("./data/title_basic.csv")

In [6]:
title_basic

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0035423,movie,Kate & Leopold,Kate & Leopold,0,2001,\N,118,"Comedy,Fantasy,Romance",6.4,76369
1,tt0036606,movie,"Another Time, Another Place","Another Time, Another Place",0,1983,\N,118,"Drama,War",6.5,243
2,tt0061876,movie,The Commissar,Komissar,0,1987,\N,110,"Drama,War",7.5,1303
3,tt0062181,movie,Rece do góry,Rece do góry,0,1981,\N,76,Drama,6.6,284
4,tt0064820,movie,The Plot Against Harry,The Plot Against Harry,0,1989,\N,81,Comedy,6.8,276
...,...,...,...,...,...,...,...,...,...,...,...
69427,tt9905462,movie,Pengalila,Pengalila,0,2019,\N,111,Drama,8.8,550
69428,tt9906644,movie,Manoharam,Manoharam,0,2019,\N,122,"Comedy,Drama",6.9,301
69429,tt9911196,movie,De Beentjes van Sint-Hildegard,De Beentjes van Sint-Hildegard,0,2020,\N,103,"Comedy,Drama",7.8,496
69430,tt9911774,movie,Padmavyuhathile Abhimanyu,Padmavyuhathile Abhimanyu,0,2019,\N,130,Drama,8.0,262


In [7]:
from sklearn.feature_extraction.text import CountVectorizer

In [8]:
count_vect = CountVectorizer(min_df=0, ngram_range=(1, 2))
genre_mat = count_vect.fit_transform(title_basic["genres"])
print(genre_mat.shape)

(69432, 252)


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

genre_sim = cosine_similarity(genre_mat, genre_mat)
print(genre_sim.shape)
print(genre_sim[:2])

(69432, 69432)
[[1.         0.         0.         0.         0.4472136  0.25819889
  0.         0.         0.         0.         0.51639778 0.
  0.         0.2        0.         0.         0.         0.
  0.4472136  0.         0.         0.4472136  0.         0.
  0.         0.         0.         0.6        0.2        0.
  0.         0.25819889 0.25819889 0.4472136  0.         0.
  0.         0.         0.25819889 0.25819889 0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.2        0.2        0.         0.51639778 0.
  0.         0.4472136  0.         0.4        0.25819889 0.4
  0.6        0.         0.2        0.2        0.         0.
  0.4472136  0.25819889 0.         0.2        0.         0.2
  0.         0.4        0.         0.2        0.         0.
  0.25819889 0.         0.         0.         0.         0.4472136
  0.         0.25819889 0.25819889 0.4472136  0.         0.4472136
  0.         0.4472136  0.         0.         0.         0.25

In [10]:
genre_sim_sorted_index = genre_sim.argsort()[:, ::-1]
genre_sim_sorted_index[:1]

array([[    0,   790,  9648, 53486, 53255, 10126, 67914, 49044, 49032,
        49001, 25000, 12831, 48419, 13434, 47359, 13707, 13952, 14052,
        46737,  9581,  8853,  8267, 57974,  6443,  6452, 58873,  1305,
        58660, 58484,  7038,  7141, 28152,  7226, 57541, 67427,  7572,
        57120,  7726, 56705, 14255, 45628, 27609, 24796, 37450, 19811,
        36795, 36458, 36146, 20781, 29952, 21382, 34200, 22222, 33097,
        22887, 22974, 31798, 31796, 31393, 24069, 37538, 37609, 19631,
        16689, 45592, 45421, 45017, 15505, 15537, 43893, 43118, 16810,
        38933, 29301, 16929, 42050, 41785, 41649, 40436, 18188,  6435,
        10054,  3705, 66923, 62828,  5317,  2006,  4486,  1587,  1523,
         2299, 61950,  5622,  2473,  2495, 67144, 65075,  3696, 61657,
         4827, 63582,  3439,  3687,  4018,  5805,  6173, 66558, 26266,
        66620, 59702, 60001, 63153, 40978, 40759,  3690, 17809, 16175,
        40538, 64849, 18350, 44855, 18455, 44902, 39351, 18638, 64814,
      

In [11]:
# df          , sorted_ind              , title_name                       , top_n=10
# movies_df     genre_sim_sorted_ind      The Godfather <- 기준 영화         추출 영화 개수
def find_sim_movie(df, sorted_ind, title_name, top_n=10):

    # 인자로 입력된 movies_df DataFrame에서 'title' 컬럼이 입력된 title_name 값인 DataFrame추출
    title_movie = df[df["primaryTitle"] == title_name]
    print(title_movie)
    # title_named을 가진 DataFrame의 index 객체를 ndarray로 반환하고
    # sorted_ind 인자로 입력된 genre_sim_sorted_ind 객체에서 유사도 순으로 top_n 개의 index 추출
    title_index = title_movie.index.values
    similar_indexes = sorted_ind[title_index, :(top_n)]

    # 추출된 top_n index들 출력. top_n index는 2차원 데이터 임.
    # dataframe에서 index로 사용하기 위해서 1차원 array로 변경
    print(similar_indexes)
    similar_indexes = similar_indexes.reshape(-1)

    return df.iloc[similar_indexes]

In [12]:
similar_movies = find_sim_movie(title_basic, genre_sim_sorted_index, "Star Trek", 10)
similar_movies[["primaryTitle", "averageRating", "numVotes"]]

          tconst titleType primaryTitle originalTitle  isAdult  startYear endYear runtimeMinutes                   genres  averageRating  numVotes
29442  tt0796366     movie    Star Trek     Star Trek        0       2009      \N            127  Action,Adventure,Sci-Fi            7.9    565829
[[ 2721 40602 11114 38380 24970 26959 68981 33063 38232 38158]]


,primaryTitle,averageRating,numVotes
2721,Aliens,8.3,624087
40602,Jupiter Ascending,5.3,172245
11114,The Fifth Element,7.7,416584
38380,Battleship,5.8,229748
24970,Transformers,7.0,581372
26959,Captain America: The First Avenger,6.9,703172
68981,Hornet,1.8,175
33063,Transformers: Revenge of the Fallen,6.0,373834
38232,The Wolverine,6.7,414791
38158,Predators,6.4,205458


In [13]:
C = title_basic["averageRating"].mean()
m = title_basic["numVotes"].quantile(0.6)  # 상위 0.6 값 추출
print("C:", round(C, 3), "m:", round(m, 3))

C: 5.89 m: 770.0


In [14]:
percentile = 0.6
m = title_basic["numVotes"].quantile(percentile)
C = title_basic["averageRating"].mean()


# record : movies_df
def weighted_vote_average(record):
    v = record["numVotes"]
    R = record["averageRating"]

    return ((v / (v + m)) * R) + ((m / (m + v)) * C)


title_basic["weighted_vote"] = title_basic.apply(weighted_vote_average, axis=1)

In [15]:
title_basic[["primaryTitle", "averageRating", "weighted_vote", "numVotes"]].sort_values(
    "weighted_vote", ascending=False
)[:10]

,primaryTitle,averageRating,weighted_vote,numVotes
8904,The Shawshank Redemption,9.3,9.298807,2200571
49742,CM101MMXI Fundamentals,9.2,9.141189,42563
64687,Love in Kilnerry,9.9,9.077992,2986
60691,Aynabaji,9.2,9.071010,18987
46305,Wheels,9.2,9.063309,17874
27425,The Dark Knight,9.0,8.998903,2182161
8832,Pulp Fiction,8.9,8.898660,1729296
13919,The Lord of the Rings: The Return of the King,8.9,8.898519,1564104
8052,Schindler's List,8.9,8.897983,1148400
66826,Peranbu,9.1,8.892104,11118


In [17]:
def find_sim_movie(df, sorted_ind, title_name, top_n=10):
    title_movie = df[df["primaryTitle"] == title_name]
    title_index = title_movie.index.values

    # top_n의 2배에 해당하는 쟝르 유사성이 높은 index 추출
    similar_indexes = sorted_ind[title_index, : (top_n * 2)]
    similar_indexes = similar_indexes.reshape(-1)

    # 기준 영화 index는 제외
    similar_indexes = similar_indexes[similar_indexes != title_index]

    # top_n의 2배에 해당하는 후보군에서 weighted_vote 높은 순으로 top_n 만큼 추출
    # similar_indexes : 20개
    return df.iloc[similar_indexes].sort_values("weighted_vote", ascending=False)[
        :top_n
    ]


similar_movies = find_sim_movie(title_basic, genre_sim_sorted_index, "Interstellar", 5)
similar_movies[["primaryTitle", "averageRating", "weighted_vote", "genres"]]

,primaryTitle,averageRating,weighted_vote,genres
54496,The Martian,8.0,7.997739,"Adventure,Drama,Sci-Fi"
28050,I Am Legend,7.2,7.198466,"Adventure,Drama,Sci-Fi"
27467,The Astronaut Farmer,6.3,6.285923,"Adventure,Drama,Sci-Fi"
4911,Es ist nicht leicht ein Gott zu sein,6.5,6.231582,"Adventure,Drama,Sci-Fi"
67012,Antariksham 9000 kmph,6.1,6.006290,"Adventure,Drama,Sci-Fi"


---

In [ ]:
rating_scales = rating_scale[:5000000]

In [ ]:
ratings_matrix = rating_scales.pivot_table("rating", index="userId", columns="movieId")

In [ ]:
ratings_matrix.fillna(0, inplace=True)

In [ ]:
ratings_matrix.to_csv("./data/ratings_matrix.csv", index=False)

In [ ]:
ratings_matrix = pd.read_csv("./data/ratings_matrix.csv")
ratings_matrix.head()

In [ ]:
ratings_matrix.shape

In [45]:
movie

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
58093,193876,The Great Glinka (1946),(no genres listed)
58094,193878,Les tribulations d'une caissière (2011),Comedy
58095,193880,Her Name Was Mumu (2016),Drama
58096,193882,Flora (2017),Adventure|Drama|Horror|Sci-Fi


In [ ]:
rating = pd.read_csv("./data/ratings.csv")
del rating

In [48]:
rating.drop("timestamp", axis=1, inplace=True)

In [49]:
rating

,userId,movieId,rating
0,1,307,3.5
1,1,481,3.5
2,1,1091,1.5
3,1,1257,4.5
4,1,1449,4.5
...,...,...,...
27753439,283228,8542,4.5
27753440,283228,8712,4.5
27753441,283228,34405,4.5
27753442,283228,44761,4.5


In [50]:
rating_movies = pd.merge(rating, movie, on="movieId")

In [51]:
rating_movies

,userId,movieId,rating,title,genres
0,1,307,3.5,Three Colors: Blue (Trois couleurs: Bleu) (1993),Drama
1,6,307,4.0,Three Colors: Blue (Trois couleurs: Bleu) (1993),Drama
2,56,307,4.0,Three Colors: Blue (Trois couleurs: Bleu) (1993),Drama
3,71,307,5.0,Three Colors: Blue (Trois couleurs: Bleu) (1993),Drama
4,84,307,3.0,Three Colors: Blue (Trois couleurs: Bleu) (1993),Drama
...,...,...,...,...,...
27753439,282403,167894,1.0,Stranglehold (1994),Action
27753440,282732,161572,3.5,The Great Houdini (1976),Drama
27753441,283000,117857,3.5,Hotline (2014),Documentary
27753442,283000,133409,3.5,Barnum! (1986),(no genres listed)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

item_sim = cosine_similarity(ratings_matrix, ratings_matrix)

item_sim_df = pd.DataFrame(
    data=item_sim, index=ratings_matrix.columns, columns=ratings_matrix.columns
)
print(item_sim_df.shape)
item_sim_df.head()